# 4 Inferenz auf einem ganzen Plan

Der Plan wird in überlappende Kacheln zerlegt (gleiche Schneide-Funktion wie für die Trainingsausschnitte), jede Kachel wird vorhergesagt, und die Detektionen werden in Scanpixel zurückgerechnet (`plan_detektieren` in `grenzpunkte.py`).
Danach Vergleich mit den manuell geklickten Punkten, Tabelle mit LV95-Koordinaten (z. B. für QGIS) und Übersichtsplot.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.path import Path
from rasterio.enums import Resampling

from grenzpunkte import DATENSATZ, ZIEL_DPI, modell_laden, plan_detektieren, zuordnen

PLAN = "H_058"

## Ganzen Plan vorhersagen

In [ ]:
plan = pd.read_csv(f"{DATENSATZ}/plaene.csv").query("status == 'ok'").set_index("plan_id").loc[PLAN]
faktor = ZIEL_DPI / plan["scan_dpi"]
detektionen = plan_detektieren(modell_laden(), PLAN, plan["scan_dpi"])[:, :2]

## Mit den Labels vergleichen
Abstände in 600-DPI-Pixeln wie in der Auswertung. Fehldetektionen zählen nur innerhalb des Digitalisierungsperimeters, denn ausserhalb wurde nichts geklickt.

In [ ]:
labels = pd.read_csv(f"{DATENSATZ}/labels/{PLAN}.csv")[["x_px", "y_px"]].values
paare = zuordnen(detektionen * faktor, labels * faktor)
treffer_det = [i for i, _, _ in paare]
fehl = np.delete(detektionen, treffer_det, axis=0)
verpasst = np.delete(labels, [j for _, j, _ in paare], axis=0)

with open(f"{DATENSATZ}/perimeter_px.json") as f:
    huelle = np.array(json.load(f)[PLAN]["digitalisierungs_huelle_5m"])
fehl_innen = fehl[Path(huelle).contains_points(fehl)]
fehler_cm = np.array([a for _, _, a in paare]) / faktor * plan["bodenpixel_m"] * 100

titel = (f"{PLAN}: {len(paare)} Treffer, {len(fehl_innen)} Fehldetektionen im Perimeter "
         f"(+{len(fehl) - len(fehl_innen)} ausserhalb), {len(verpasst)} verpasst | "
         f"Lage Median {np.median(fehler_cm):.1f} cm, p95 {np.percentile(fehler_cm, 95):.1f} cm")
print(titel)

## Detektionen als Tabelle speichern
E, N in LV95 (Hasle: vorläufiges CAD-System), über den GeoTransform des GeoTIFFs.

In [ ]:
with rasterio.open(f"{DATENSATZ}/geotiff/{PLAN}.tif") as geotiff:
    E, N = geotiff.transform * (detektionen[:, 0], detektionen[:, 1])
    # für die Anzeige in halber Auflösung lesen (voll braucht matplotlib über 10 GB RAM)
    scan = geotiff.read(out_shape=(3, geotiff.height // 2, geotiff.width // 2), resampling=Resampling.average)
    breite, hoehe = geotiff.width, geotiff.height

status = np.full(len(detektionen), "fehldetektion", dtype=object)
status[treffer_det] = "treffer"
pd.DataFrame({"x_px": detektionen[:, 0].round(2), "y_px": detektionen[:, 1].round(2), "E": E.round(3), "N": N.round(3),
              "status": status}).to_csv(f"data/auswertung/{PLAN}_detektionen.csv", index=False)

## Übersichtsplot
`+` = manuell geklickt, `×` = Modell; Kreise markieren Fehldetektionen (orange) und verpasste Punkte (blau).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 16))
ax.imshow(scan.transpose(1, 2, 0), extent=(0, breite, hoehe, 0))  # Achsen in Scanpixeln des Originals
ax.plot(*np.vstack([huelle, huelle[:1]]).T, "--", c="gray", lw=1, label="Digitalisierungsperimeter")
ax.scatter(labels[:, 0], labels[:, 1], marker="+", c="lime", s=30, lw=1, label="manuell geklickt")
ax.scatter(detektionen[:, 0], detektionen[:, 1], marker="x", c="red", s=20, lw=1, label="Modell")
ax.scatter(fehl_innen[:, 0], fehl_innen[:, 1], s=250, facecolors="none", edgecolors="orange", lw=1.5,
           label="Fehldetektion (im Perimeter)")
ax.scatter(verpasst[:, 0], verpasst[:, 1], s=250, facecolors="none", edgecolors="blue", lw=1.5, label="verpasst")
ax.set_title(titel, fontsize=9)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(f"data/auswertung/{PLAN}_plan.png", dpi=150)
plt.show()